In [4]:
import math

class Point:
    def __init__(self, x=0, y=0):
        self._x = x
        self._y = y

    def get_x(self):
        return self._x

    def set_x(self, x):
        self._x = x

    def get_y(self):
        return self._y

    def set_y(self, y):
        self._y = y

    def __eq__(self, other):
        if isinstance(other, Point):
            return self._x == other.get_x() and self._y == other.get_y()
        return False

    def  __str__(self):
         return f'Point({self.get_x()}, {self.get_y()})'

In [5]:
class Segment:
    def __init__(self, point1, point2):
        self._point1 = point1
        self._point2 = point2

    def is_parallel(self, other_segment):
        # Get the coordinates of the points
        x1, y1 = self._point1.get_x(), self._point1.get_y()
        x2, y2 = self._point2.get_x(), self._point2.get_y()
        x3, y3 = other_segment._point1.get_x(), other_segment._point1.get_y()
        x4, y4 = other_segment._point2.get_x(), other_segment._point2.get_y()

        # Calculate slopes
        slope1 = (y2 - y1) / (x2 - x1) if x2 != x1 else float('inf')  # Handle vertical line
        slope2 = (y4 - y3) / (x4 - x3) if x4 != x3 else float('inf')  # Handle vertical line

        # Check if slopes are equal or both lines are vertical
        return slope1 == slope2

    # Other methods are unchanged
    def is_collinear(self, other_segment):
        # Get the coordinates of the points
        x1, y1 = self._point1.get_x(), self._point1.get_y()
        x2, y2 = self._point2.get_x(), self._point2.get_y()
        x3, y3 = other_segment._point1.get_x(), other_segment._point1.get_y()
        x4, y4 = other_segment._point2.get_x(), other_segment._point2.get_y()

        # Calculate slopes
        slope1 = (y2 - y1) / (x2 - x1) if x2 != x1 else float('inf')  # Handle vertical line
        slope2 = (y4 - y3) / (x4 - x3) if x4 != x3 else float('inf')  # Handle vertical line

        # Calculate intercepts
        intercept1 = y1 - slope1 * x1 if slope1 != float('inf') else x1  # Handle vertical line
        intercept2 = y3 - slope2 * x3 if slope2 != float('inf') else x3  # Handle vertical line

        # Check if segments are collinear
        return self.is_parallel(other_segment) and intercept1 == intercept2

    def is_point_on_segment(self, point):
        x, y = point.get_x(), point.get_y()
        new_segment = Segment(self._point1, point)

        if (self._point1 == point) or (self._point2 == point):
            return True
        
        if not self.is_collinear(new_segment):
            return False
        
        x1, y1 = self._point1.get_x(), self._point1.get_y()
        x2, y2 = self._point2.get_x(), self._point2.get_y()
    
        # Check if the point is collinear with the segment endpoints
        if ((x1 <= x <= x2 or x2 <= x <= x1) and
                (y1 <= y <= y2 or y2 <= y <= y1)):
            return True
    
        return False

    #returns none if segments don't intersect, Point if they intersect  incl. collinear case
    def get_intersection_point(self, other_segment):
        x1, y1 = self._point1.get_x(), self._point1.get_y()
        x2, y2 = self._point2.get_x(), self._point2.get_y()
        x3, y3 = other_segment._point1.get_x(), other_segment._point1.get_y()
        x4, y4 = other_segment._point2.get_x(), other_segment._point2.get_y()

        # For collinear case it is enough to check 3 points
        if self.is_collinear(other_segment):
            if (self.is_point_on_segment(other_segment._point1)):
                return other_segment._point1
            if (self.is_point_on_segment(other_segment._point2)):
                return other_segment._point2
            if (other_segment.is_point_on_segment(self._point1)):
                return self._point1
            return None
            
        # Calculate intersection point using line equations
        denominator = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if denominator == 0:  # Lines are parallel
            return None
        
        px = ((x1 * y2 - y1 * x2) * (x3 - x4) - (x1 - x2) * (x3 * y4 - y3 * x4)) / denominator
        py = ((x1 * y2 - y1 * x2) * (y3 - y4) - (y1 - y2) * (x3 * y4 - y3 * x4)) / denominator

        # Check if intersection point lies within both segments
        if (min(x1, x2) <= px <= max(x1, x2) and
            min(y1, y2) <= py <= max(y1, y2) and
            min(x3, x4) <= px <= max(x3, x4) and
            min(y3, y4) <= py <= max(y3, y4)):
            return Point(px, py)
        
        return None

In [6]:
class Triangle:
    def __init__(self, point1, point2, point3):
        self._point1 = point1
        self._point2 = point2
        self._point3 = point3
        self._segment1 = Segment(point1, point2)
        self._segment2 = Segment(point2, point3)
        self._segment3 = Segment(point1, point3)

    def is_point_inside(self, point):
        # Calculate barycentric coordinates of the point with respect to the triangle's vertices
        x, y = point.get_x(), point.get_y()
        x1, y1 = self._point1.get_x(), self._point1.get_y()
        x2, y2 = self._point2.get_x(), self._point2.get_y()
        x3, y3 = self._point3.get_x(), self._point3.get_y()

        denominator = ((y2 - y3) * (x1 - x3) + (x3 - x2) * (y1 - y3))
        if denominator == 0:  # Triangle is degenerate, points are collinear
            #TODO: fix degenerate case
            if (self._segment1.is_point_on_segment(point) or \
                self._segment2.is_point_on_segment(point) or \
                self._segment3.is_point_on_segment(point)):
                return True
            return False

        # Calculate barycentric coordinates
        alpha = ((y2 - y3) * (x - x3) + (x3 - x2) * (y - y3)) / denominator
        beta = ((y3 - y1) * (x - x3) + (x1 - x3) * (y - y3)) / denominator
        gamma = 1.0 - alpha - beta

        # Check if the point lies inside the triangle or on its sides
        return 0 <= alpha <= 1 and 0 <= beta <= 1 and 0 <= gamma <= 1

triangle = Triangle(Point(1, 1), Point(2, 2), Point(3, 3))
point_inside = Point(1, 1)
point_on_side = Point(3, 1)
point_outside = Point(6, 2)

print("Is point_inside inside the triangle:", triangle.is_point_inside(point_inside))
print("Is point_on_side inside the triangle:", triangle.is_point_inside(point_on_side))
print("Is point_outside inside the triangle:", triangle.is_point_inside(point_outside))

Is point_inside inside the triangle: True
Is point_on_side inside the triangle: False
Is point_outside inside the triangle: False


In [ ]:
# And implement the algorithm from the Lavesh's thesis
    

In [10]:
# here will be the function related to obstacles

In [18]:
# I also have to implement metaheuristic to find a proper path